# Example usage of SimPyJobShop

## Create a Problem 
For illustration purposes, create a simple parallel machine problem with 5 jobs
 and 4 machines. The job durations are generated from a Poisson distribution 
 with a random mean. The objective is to minimize the makespan. The problem 
 is specified in the `ParallelMachines` class, which inherits from the `Problem` class.
In this class, the `concrete_model` method creates the model, and the
 `distribution_data` method generates the job durations.

In [1]:
from typing import Any, Dict, Tuple

import numpy as np

from pyjobshop import Model
from simpyjobshop.DiscreteRV import DiscreteRV, SeededPoisson
from simpyjobshop.problems import Problem


class ParallelMachines(Problem):

    @staticmethod
    def concrete_model(data: Dict[str, Any]) -> Model:
        # Unpack data
        num_jobs = data["num_jobs"]
        num_machines = data["num_machines"]
        durations = [
            [data[f"duration_{j}_on_{m}"] for j in range(num_jobs)]
            for m in range(num_machines)
            ]
        objective_weights = {"weight_makespan": data["weight_makespan"]}

        # Create model
        model = Model()
        jobs = [model.add_job() for _ in range(num_jobs)]
        tasks = [model.add_task(job) for job in jobs]
        machines = [model.add_machine() for _ in range(num_machines)]

        # Add modes to the model
        for m, machine in enumerate(machines):
            for task, duration in zip(tasks, durations[m], strict=True):
                model.add_mode(task, machine, duration=duration)

        # Set objective
        model.set_objective(**objective_weights)

        return model

    def distribution_data(
        self, seed: int = 0
        ) -> Tuple[Dict[str, DiscreteRV], Dict[str, int]]:
        distributions: Dict[str, DiscreteRV] = {}
        constants: Dict[str, int] = {}

        num_jobs = 10
        num_machines = 4
        max_mean_dur = 20

        # job durations
        np.random.seed(seed)  # for reproducibility
        gen: DiscreteRV
        for machine_idx in range(num_machines):
            for job_idx in range(num_jobs):
                mean_job_duration = np.random.randint(max_mean_dur)
                job_seed = job_idx + machine_idx * num_jobs
                gen = SeededPoisson(lam=mean_job_duration, loc=1, seed=job_seed)
                distributions[f"duration_{job_idx}_on_{machine_idx}"] = gen

        # store constants
        constants["num_jobs"] = num_jobs
        constants["num_machines"] = num_machines
        constants["weight_makespan"] = 1

        return distributions, constants


## Solve the Problem

Let us now solve the problem using CP-SAT. Two approaches are applied: one approach
that ignores the uncertainty and assumes all random variables can be represented
by their means, and a second approach that uses a standard simheuristic. 

### Load the simheuristics

In [2]:
from experiments.utils.SimheuristicSpec import SimheuristicSpec
from simpyjobshop.simheuristics import (
    StandardSimheuristicConfig,
    deterministic_optimization,
    standard_simheuristic, DeterministicOptimizationConfig,
    )

problem_seed = 6
problem = ParallelMachines(seed=problem_seed)

exp_config = {
    "time_limit": 60,
    "num_workers": 1,  # (for reproducibility)
    "use_wandb": False,
    # only relevant for cli_run_experiments.py and submit_slurm_job.py:
    "num_rand_experiments": None,
    "num_parallel_instances": None,
    "num_sims_for_true_expec_objective": None,
    }

# Simheuristic configurations
det_opt_config: DeterministicOptimizationConfig = {
    "det_repr": "mean",
    }
stand_simh_config: StandardSimheuristicConfig = {
    "det_repr": "mean",
    "num_sims": 20,
    "max_size_elite_set": 5,
    "frac_budget_before_sims": 10 / exp_config["time_limit"],
    "frac_budget_final_elites_sim": 5 / exp_config["time_limit"],
    }

simheuristics: list[SimheuristicSpec] = [
    SimheuristicSpec("std_simh", standard_simheuristic, stand_simh_config),
    SimheuristicSpec("det_opt", deterministic_optimization, det_opt_config),
    ]

### Solve the problem

In [3]:

from simpyjobshop.utils import plot_gantt_chart

for simheuristic in simheuristics:
    callback, last_CP_results, durations = simheuristic.fun(
        problem,
        simheuristic.config,
        exp_config,
        )

    # Find elite
    elite_solutions = callback.solutions
    if elite_solutions.all_simulated():
        elite = elite_solutions.get_best_mean_solution()
    elif elite_solutions.none_simulated():
        elite = elite_solutions.get_best_deterministic_solution()
    else:
        raise Exception(
            "Some elites are simulated and some not. This is not expected."
            )

    # Simulate the elite solution
    num_sims = 500 - elite.simulator.num_sims
    elite.simulator.simulate(num_sims=num_sims)

    # Plot elite
    title=(
        f"Solution found with {simheuristic.name} with "
        f"deterministic objective value {elite.objective} and "
        f"mean objective value {elite.simulator.mean}"
        f" (num. of sims. = {elite.simulator.num_sims})"
    )
    plot_gantt_chart(elite.solution, problem, title)
